# Практическая работа № 2 (пример выполнения)
## Иерархия игровых сущностей: ABC, протоколы, dataclass, миксины

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 2 — Углублённое ООП на Python  
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине  
**Формат:** демонстрационный блокнот с полным разбором задания  
**Цель:** показать, как развить проект из задания №1, добавив абстрактные классы, протоколы, `dataclass`-конфигурации и миксин логирования

---

## 📖 Контекст: что уже есть в проекте

В практической работе №1 мы создали процедурный прототип и провели его рефакторинг в ООП. В проекте уже есть:

| Класс | Ответственность |
|---|---|
| `Character` | Базовый персонаж (имя, описание) |
| `Detective` | Граф Аверин: энергия, улики, Плеть |
| `Div` | Кузя: формы, лояльность, разведка |
| `Location` | Локация: улики, персонажи |
| `Clue` | Улика: название, описание, важность |
| `Game` | Игровой цикл |
| `UI` | Ввод/вывод |

**Проблема:** классы `Character`, `Detective`, `Div` дублируют поведение, а способности (движение, отрисовка, коллизии) не выделены в явные контракты. Кроме того, конфигурации (настройки игры, параметры персонажей) задаются прямо в коде, а логирование отсутствует.

## 🎯 Что демонстрирует этот блокнот

- **ABC `GameObject`** — явный базовый класс для всех игровых сущностей.
- **Протоколы `Movable`, `Drawable`, `Collidable`** — структурные интерфейсы для способностей.
- **`@dataclass`** — конфигурации игры и персонажей вместо магических чисел.
- **Миксин `LoggingMixin`** — переиспользуемое логирование действий.
- **Юнит-тесты** — проверка абстрактных классов, протоколов, dataclass и миксина.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# 1. Абстрактный класс `GameObject`

## Теория: зачем нужен ABC

**Абстрактный базовый класс** — это класс, который нельзя инстанцировать напрямую. Он задаёт **контракт**: набор методов, которые обязаны реализовать подклассы. Если подкласс забыл реализовать хотя бы один абстрактный метод, Python выбросит `TypeError` при попытке создать экземпляр.[reference:0]

**Модуль `abc`** предоставляет:
- `ABC` — базовый класс для создания ABC;
- `@abstractmethod` — декоратор, помечающий метод как абстрактный.[reference:1]

**Пример из игровой разработки:** базовый класс `GameObject` требует, чтобы все игровые объекты умели обновляться (`update`) и отрисовываться (`draw`). Это гарантирует, что игровой цикл может единообразно работать с любыми объектами.[reference:2]

**Для углублённого изучения:**
- [Real Python: Implementing Interfaces in Python: ABCs and Protocols](https://realpython.com/python-interface/)
- [Python docs: abc — Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [PEP 3119 — Introducing Abstract Base Classes](https://peps.python.org/pep-3119/)

In [ ]:
%%writefile mystic_detective/core/game_object.py
"""Абстрактный базовый класс для всех игровых сущностей."""

from abc import ABC, abstractmethod


class GameObject(ABC):
    """Базовый класс для всего, что существует в игровом мире.

    Каждый игровой объект обязан:
    - иметь уникальный идентификатор;
    - уметь обновляться каждый кадр (update);
    - уметь отрисовываться (draw).
    """

    _next_id: int = 0

    def __init__(self, name: str):
        self._id = GameObject._next_id
        GameObject._next_id += 1
        self._name = name
        self._active = True

    @property
    def id(self) -> int:
        return self._id

    @property
    def name(self) -> str:
        return self._name

    @property
    def active(self) -> bool:
        return self._active

    def deactivate(self) -> None:
        """Пометить объект как неактивный (удалить из игры)."""
        self._active = False

    @abstractmethod
    def update(self, dt: float) -> None:
        """Обновить состояние объекта за время dt (секунды)."""
        ...

    @abstractmethod
    def draw(self) -> str:
        """Вернуть строковое представление для отрисовки."""
        ...

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(id={self._id}, name={self._name!r})"


### Проверка: ABC нельзя инстанцировать

In [ ]:
import sys
sys.path.insert(0, 'mystic_detective')

from core.game_object import GameObject

try:
    GameObject("test")
except TypeError as e:
    print(f"✅ Ожидаемая ошибка: {e}")

---
# 2. Протоколы `Movable`, `Drawable`, `Collidable`

## Теория: структурная типизация

**Протокол** (`typing.Protocol`) — это способ описать интерфейс **без наследования**. Класс считается совместимым с протоколом, если он **имеет** нужные методы и атрибуты, независимо от того, наследуется он от протокола или нет.[reference:3]

**Разница между ABC и Protocol:**

| Критерий | ABC | Protocol |
|---|---|---|
| Требует наследования | Да | Нет |
| Проверка во время выполнения | Да | Только с `@runtime_checkable` |
| Статическая проверка | Да | Да |
| Когда использовать | Когда вы владеете иерархией | Когда важно «что умеет», а не «чем является» |[reference:4]

**Пример из игры:** способности объектов (движение, отрисовка, коллизии) — это роли, а не типы. Детектив может быть `Movable` и `Drawable`, но не быть `Collidable`. Протоколы позволяют описать эти роли гибко.

**Для углублённого изучения:**
- [PEP 544 — Protocols: Structural subtyping](https://peps.python.org/pep-0544/)
- [Real Python: Structural Subtyping with Protocols](https://realpython.com/python-protocol/)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)

In [ ]:
%%writefile mystic_detective/core/protocols.py
"""Протоколы способностей игровых объектов."""

from typing import Protocol, runtime_checkable


@runtime_checkable
class Movable(Protocol):
    """Всё, что может двигаться."""

    def move(self, dx: float, dy: float) -> None:
        """Переместиться на (dx, dy)."""
        ...


@runtime_checkable
class Drawable(Protocol):
    """Всё, что можно отрисовать."""

    def draw(self) -> str:
        """Вернуть строковое представление для отрисовки."""
        ...


@runtime_checkable
class Collidable(Protocol):
    """Всё, что может участвовать в коллизиях."""

    @property
    def collision_radius(self) -> float:
        """Радиус для проверки коллизий."""
        ...

    def collides_with(self, other: "Collidable") -> bool:
        """Проверить коллизию с другим объектом."""
        ...


### Проверка протоколов

In [ ]:
from core.protocols import Movable, Drawable, Collidable


class SimpleObject:
    def move(self, dx, dy):
        print(f"Moved by ({dx}, {dy})")

    def draw(self):
        return "SimpleObject"


obj = SimpleObject()
print(f"obj is Movable: {isinstance(obj, Movable)}")
print(f"obj is Drawable: {isinstance(obj, Drawable)}")
print(f"obj is Collidable: {isinstance(obj, Collidable)}")
print(f"int is Drawable: {isinstance(42, Drawable)}")

---
# 3. `@dataclass` для конфигураций и компонентов

## Теория: dataclass

`@dataclass` автоматически генерирует `__init__`, `__repr__` и `__eq__` на основе аннотаций типов. Это устраняет шаблонный код для классов-данных.[reference:5]

**Параметры `@dataclass`:**

| Параметр | По умолчанию | Что делает |
|---|---|---|
| `frozen` | `False` | Запрещает изменение после создания |
| `slots` | `False` | Добавляет `__slots__` для экономии памяти |

**`frozen=True`** особенно полезен для конфигураций: они не должны меняться во время выполнения. Замороженные dataclass также хешируемы, что позволяет использовать их как ключи словарей.[reference:6]

**`field(default_factory=...)`** решает проблему изменяемых значений по умолчанию (списков, словарей).

**Для углублённого изучения:**
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/)
- [Real Python: Data Classes in Python 3.7+](https://realpython.com/python-data-classes/)

In [ ]:
%%writefile mystic_detective/core/config.py
"""Конфигурации игры и персонажей."""

from dataclasses import dataclass, field


@dataclass(frozen=True)
class GameConfig:
    """Общие настройки игры."""

    title: str = "Тайна исчезнувшего дива"
    max_energy: int = 100
    investigate_cost: int = 10
    whip_cost: int = 20
    rest_amount: int = 5


@dataclass(frozen=True)
class DetectiveConfig:
    """Параметры детектива."""

    name: str = "Граф Гермес Аверин"
    energy: int = 100
    max_energy: int = 100


@dataclass(frozen=True)
class DivConfig:
    """Параметры дива-напарника."""

    name: str = "Кузя"
    form: str = "кот"
    loyalty: int = 50
    forms: tuple = ("кот", "юноша", "ворон")


@dataclass
class LocationConfig:
    """Конфигурация локации."""

    name: str
    description: str = ""
    clues: list = field(default_factory=list)
    characters: list = field(default_factory=list)


@dataclass
class ClueData:
    """Данные улики (компонент)."""

    name: str
    description: str = ""
    is_key: bool = False


### Проверка dataclass

In [ ]:
from core.config import GameConfig, DetectiveConfig, DivConfig, ClueData

cfg = GameConfig()
print(cfg)

det = DetectiveConfig()
print(det)

div = DivConfig()
print(div)

clue = ClueData("Записка", "План Пустоши", is_key=True)
print(clue)

# frozen: попытка изменить вызовет ошибку
try:
    cfg.max_energy = 200
except Exception as e:
    print(f"✅ Ожидаемая ошибка: {type(e).__name__}: {e}")

---
# 4. Миксин `LoggingMixin`

## Теория: миксины

**Миксин** — это класс, предназначенный для добавления функциональности другим классам через множественное наследование. Миксины:
- не предназначены для самостоятельного использования;
- не имеют собственного состояния (`__init__`);
- обычно определяют один-два метода.[reference:7]

**Ключевое правило:** миксин должен вызывать `super()` для кооперативного наследования.

**Зачем это в игре:** логирование действий (атака, перемещение, поиск улик) полезно для отладки и аналитики. Миксин позволяет добавить логирование любому классу одной строкой.

**Для углублённого изучения:**
- [Python Cookbook: Extending Classes with Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html)
- [Real Python: Multiple Inheritance and Mixins](https://realpython.com/inheritance-composition-python/)

In [ ]:
%%writefile mystic_detective/core/mixins.py
"""Миксины для переиспользования поведения."""

import logging


class LoggingMixin:
    """Миксин: добавляет логирование действий объекта.

    Использование:
        class MyClass(LoggingMixin, SomeBase):
            def do_something(self):
                self.log("Doing something")
    """

    __slots__ = ()

    def log(self, message: str) -> None:
        """Записать сообщение в лог."""
        logger = logging.getLogger(self.__class__.__name__)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                "[%(name)s] %(levelname)s: %(message)s"
            )
            handler.setFormatter(formatter)
            logger.addHandler(handler)
            logger.setLevel(logging.INFO)
        logger.info(message)

    def log_warning(self, message: str) -> None:
        """Записать предупреждение."""
        logging.getLogger(self.__class__.__name__).warning(message)

    def log_error(self, message: str) -> None:
        """Записать ошибку."""
        logging.getLogger(self.__class__.__name__).error(message)


### Проверка миксина

In [ ]:
from core.mixins import LoggingMixin


class TestClass(LoggingMixin):
    def do_work(self):
        self.log("Working...")


obj = TestClass()
obj.do_work()
print("✅ Логирование работает")

---
# 5. Интеграция: обновление `Detective` и `Div`

## Теория: как совместить ABC, протоколы и миксины

Теперь мы можем обновить существующие классы:
- `Detective` наследуется от `GameObject` (ABC) и `LoggingMixin`;
- реализует протокол `Drawable` (метод `draw`);
- использует `DetectiveConfig` из dataclass.

**Порядок наследования:** `class Detective(LoggingMixin, GameObject)`. Миксин идёт первым, чтобы его методы были доступны в MRO.

**Для углублённого изучения:**
- [Real Python: Method Resolution Order](https://realpython.com/python-mro/)
- [Python docs: MRO](https://docs.python.org/3/howto/mro.html)

In [ ]:
%%writefile mystic_detective/entities/detective_v2.py
"""Обновлённый Detective: наследует GameObject, использует миксин и конфиг."""

from core.game_object import GameObject
from core.mixins import LoggingMixin
from core.config import DetectiveConfig
from entities.clue import Clue
from entities.div import Div


class Detective(LoggingMixin, GameObject):
    """Граф Аверин: колдун, детектив, владелец Плети.

    Наследует:
    - LoggingMixin — логирование действий;
    - GameObject — контракт update/draw.
    """

    def __init__(
        self,
        config: DetectiveConfig | None = None,
        div: Div | None = None,
    ):
        cfg = config if config is not None else DetectiveConfig()
        super().__init__(cfg.name)
        self._energy = cfg.energy
        self._max_energy = cfg.max_energy
        self._clues: list[Clue] = []
        self._div = div
        self.log(f"Создан детектив {cfg.name}")

    @property
    def energy(self) -> int:
        return self._energy

    @property
    def max_energy(self) -> int:
        return self._max_energy

    @property
    def clues(self) -> list[Clue]:
        return list(self._clues)

    @property
    def div(self) -> Div | None:
        return self._div

    def add_clue(self, clue: Clue) -> None:
        if clue not in self._clues:
            self._clues.append(clue)
            self.log(f"Найдена улика: {clue.name}")

    def investigate(self, location_name: str, cost: int = 10) -> str:
        if self._energy < cost:
            self.log_warning("Не хватает энергии для осмотра")
            return "Не хватает энергии для осмотра."
        self._energy -= cost
        self.log(f"Осмотр локации {location_name} (энергия: {self._energy})")
        return f"{self._name} осматривает {location_name}"

    def use_whip(self, cost: int = 20) -> str:
        if self._energy < cost:
            self.log_warning("Не хватает энергии для Плети")
            return "Не хватает энергии для Плети."
        self._energy -= cost
        self.log(f"Применение Плети (энергия: {self._energy})")
        return "Плеть рассекает воздух!"

    def rest(self, amount: int = 20) -> None:
        self._energy = min(self._max_energy, self._energy + amount)
        self.log(f"Отдых: энергия восстановлена до {self._energy}")

    def update(self, dt: float) -> None:
        """Обновление состояния (реализация абстрактного метода)."""
        pass

    def draw(self) -> str:
        """Отрисовка (реализация абстрактного метода)."""
        return f"[Detective] {self._name} (HP: {self._energy}/{self._max_energy})"


### Проверка интеграции

In [ ]:
from entities.div import Div
from entities.detective_v2 import Detective
from core.config import DetectiveConfig, DivConfig
from core.protocols import Drawable, Movable

div = Div("Кузя", form="кот", loyalty=50)
det = Detective(config=DetectiveConfig(), div=div)

print(det.draw())
print(f"isinstance(det, Drawable): {isinstance(det, Drawable)}")

det.investigate("особняк")
det.use_whip()
det.rest(5)
print(det.draw())

---
# 6. Юнит-тесты

## Теория: что тестировать

В этом задании нужно проверить:
- **ABC нельзя инстанцировать** — `GameObject("x")` должен выбрасывать `TypeError`;[reference:8]
- **Протоколы работают** — объект с нужными методами проходит `isinstance`;
- **dataclass корректен** — `frozen` запрещает изменение, `default_factory` создаёт независимые списки;
- **Миксин логирует** — вызов `log()` не падает.

**Для углублённого изучения:**
- [pytest documentation](https://docs.pytest.org/)
- [Real Python: Testing Your Code with pytest](https://realpython.com/pytest-python-testing/)

In [ ]:
%%writefile mystic_detective/tests/test_game_object.py
"""Тесты для ABC GameObject."""

import pytest
from core.game_object import GameObject


class TestGameObjectABC:
    def test_cannot_instantiate_abstract(self):
        """ABC нельзя создать напрямую."""
        with pytest.raises(TypeError):
            GameObject("test")

    def test_subclass_must_implement_methods(self):
        """Подкласс без update/draw нельзя создать."""
        class Incomplete(GameObject):
            pass

        with pytest.raises(TypeError):
            Incomplete("test")

    def test_concrete_subclass_works(self):
        """Полный подкласс создаётся и имеет id."""
        class Concrete(GameObject):
            def update(self, dt):
                pass

            def draw(self):
                return "Concrete"

        obj = Concrete("test")
        assert obj.name == "test"
        assert isinstance(obj.id, int)
        assert obj.active


In [ ]:
%%writefile mystic_detective/tests/test_protocols.py
"""Тесты для протоколов."""

from core.protocols import Movable, Drawable, Collidable


class TestProtocols:
    def test_object_with_draw_is_drawable(self):
        class WithDraw:
            def draw(self):
                return "x"

        assert isinstance(WithDraw(), Drawable)

    def test_object_without_draw_is_not_drawable(self):
        class WithoutDraw:
            pass

        assert not isinstance(WithoutDraw(), Drawable)

    def test_object_with_move_is_movable(self):
        class WithMove:
            def move(self, dx, dy):
                pass

        assert isinstance(WithMove(), Movable)

    def test_object_with_collision_is_collidable(self):
        class WithCollision:
            @property
            def collision_radius(self):
                return 1.0

            def collides_with(self, other):
                return False

        assert isinstance(WithCollision(), Collidable)


In [ ]:
%%writefile mystic_detective/tests/test_config.py
"""Тесты для dataclass-конфигураций."""

import pytest
from core.config import GameConfig, DetectiveConfig, ClueData


class TestConfig:
    def test_game_config_frozen(self):
        cfg = GameConfig()
        with pytest.raises(Exception):
            cfg.max_energy = 200

    def test_game_config_defaults(self):
        cfg = GameConfig()
        assert cfg.max_energy == 100
        assert cfg.investigate_cost == 10

    def test_clue_data_independent_lists(self):
        a = ClueData("A")
        b = ClueData("B")
        assert a.name == "A"
        assert b.name == "B"

    def test_detective_config_equality(self):
        a = DetectiveConfig()
        b = DetectiveConfig()
        assert a == b


In [ ]:
%%writefile mystic_detective/tests/test_mixins.py
"""Тесты для LoggingMixin."""

from core.mixins import LoggingMixin


class TestLoggingMixin:
    def test_log_does_not_crash(self):
        class Foo(LoggingMixin):
            pass

        foo = Foo()
        foo.log("test message")

    def test_log_warning(self):
        class Foo(LoggingMixin):
            pass

        foo = Foo()
        foo.log_warning("warning")


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-v', '--tb=short'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Код выхода: {result.returncode}")

---
# 7. Итоги и литература

## Что мы изучили

| Инструмент | Назначение | Пример в проекте |
|---|---|---|
| **ABC `GameObject`** | Явный контракт для игровых объектов | `Detective` наследует `GameObject` |
| **Protocol `Movable`** | Структурный интерфейс движения | Любой объект с `move()` |
| **Protocol `Drawable`** | Структурный интерфейс отрисовки | `Detective.draw()` |
| **Protocol `Collidable`** | Структурный интерфейс коллизий | Объекты с `collision_radius` |
| **`@dataclass`** | Конфигурации и компоненты | `GameConfig`, `DetectiveConfig` |
| **`frozen=True`** | Неизменяемые конфиги | `GameConfig` |
| **`field(default_factory)`** | Безопасные списки | `LocationConfig.clues` |
| **`LoggingMixin`** | Логирование действий | `Detective` логирует расследование |

## Практические рекомендации

1. **Используйте ABC** для базовых классов, которые вы контролируете.
2. **Используйте Protocol** для способностей и внешних зависимостей.
3. **Используйте dataclass** для конфигураций (особенно `frozen=True`).
4. **Используйте миксины** для сквозной функциональности (логирование, сериализация).
5. **Порядок наследования:** миксин первым, ABC вторым.

## Литература

**Базовая:**
- Лутц М. «Изучаем Python», главы 26–27.
- [Python docs: abc](https://docs.python.org/3/library/abc.html)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)

**Углублённая:**
- Ramalho L. «Fluent Python», 2-е изд. — главы о протоколах и дескрипторах.
- [Real Python: Implementing Interfaces in Python](https://realpython.com/python-interface/)
- [Real Python: Data Classes](https://realpython.com/python-data-classes/)
- [PEP 544 — Protocols](https://peps.python.org/pep-0544/)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/)

## Что дальше

В модуле 3 мы изучим принципы SOLID и рефакторинг. Проект «Тайна исчезнувшего дива» будет развиваться: мы выделим ответственности классов, применим инверсию зависимостей и подготовим архитектуру к паттернам проектирования.